# Homework 11: Neural Style Transfer as a Small Experimental Lab

**Student Version**

**Status:** optional / extra lab unless your instructor says otherwise  
**Release:** May 31, 2026  
**Deadline:** June 5, 2026  
**Recommended time budget:** 3-5 focused hours  
**Recommended environment:** Google Colab with GPU. CPU can work for small settings, but optimization will be slow.

This homework is a visual lab about transfer learning with a fixed pretrained CNN. You will not train VGG19. You will use VGG19 as a feature measuring tool and optimize the pixels of a generated image.

Goals:
- Understand why a pretrained VGG network can be reused without training
- Inspect feature maps from early and deeper VGG layers
- Implement content loss and the Gram matrix
- Combine style losses across several layers
- Run and interpret controlled style-transfer experiments

Minimum viable submission:
- feature-map inspection for one early and one deeper VGG layer
- working `content_loss(...)` and `gram_matrix(...)`
- working multi-layer `style_loss(...)`
- one baseline style-transfer result
- content-heavy vs style-heavy comparison
- content initialization vs noise initialization comparison
- a short result table and written answers

Suggested grading guide if this is used for credit:

| Part | Weight | What matters most |
|---|---:|---|
| Feature inspection | 15% | correct layer access, shapes, channel visualizations, interpretation |
| Content loss and Gram matrix | 25% | correct formulas, shapes, symmetry check, Gram interpretation |
| Multi-layer style loss | 20% | style targets, style loss, near-zero self-loss |
| Controlled experiments | 25% | baseline, style-weight comparison, initialization comparison |
| Summary and reflection | 15% | clear table, concise qualitative conclusions, wrap-up answers |

The goal is not to make the prettiest image. The goal is to understand what is optimized, what VGG contributes, and how content/style weights change the result.


## 0. Theory: Transfer Learning, Content, and Style

Neural style transfer uses a pretrained CNN in a slightly unusual way.

In classification, a CNN is trained to map an image to class logits. During training, the network learns reusable visual features:
- early layers respond to edges, colors, and simple textures,
- middle layers respond to repeated patterns and parts,
- deeper layers respond to larger object-like structures.

Transfer learning means we reuse these learned features for a new task instead of training a new model from scratch.

For style transfer:
- VGG19 is kept fixed,
- we do not update VGG weights,
- we optimize the pixels of a generated image,
- content loss keeps high-level structure close to the content image,
- style loss makes feature-channel correlations close to the style image.

The core idea:

```text
fixed pretrained CNN = feature measuring tool
generated image      = trainable object
```

So the network is not the final product. The final product is the optimized image.

### Feature Maps

Let a VGG layer produce activations:

```text
F_l(x): [C, H, W]
```

For one image, this means:
- `C` feature channels,
- each channel is a spatial map of size `H x W`,
- channel `c` responds to some visual pattern learned by VGG.

For content, we care about where things are. So we compare feature maps directly.

Content loss at layer `l`:

```text
L_content = mean((F_l(generated) - F_l(content))^2)
```

This says: the generated image should activate the same high-level VGG features in roughly the same spatial locations as the content image.

### Why Raw Feature Maps Are Not Enough For Style

Style is less about exact object location and more about repeated visual statistics:
- colors,
- textures,
- brush strokes,
- local patterns,
- which feature channels tend to appear together.

If we compared raw feature maps to match style, we would force the generated image to place style patterns in exactly the same locations as the style image. That is not what we want. We want the texture statistics, not the style image layout.

### Gram Matrix

Take a feature map:

```text
F: [C, H, W]
```

Flatten the spatial dimensions:

```text
F_flat: [C, H * W]
```

Each row is one feature channel written as a long vector over all spatial positions.

The Gram matrix is:

```text
G = F_flat @ F_flat.T
```

So:

```text
G[i, j] = sum over positions of F_flat[i, position] * F_flat[j, position]
```

Interpretation:
- large `G[i, j]`: channels `i` and `j` often activate together,
- small `G[i, j]`: they do not co-activate much,
- diagonal `G[i, i]`: how strongly channel `i` activates overall.

This captures style because texture is partly about which visual patterns co-occur, regardless of where exactly they occur.

We normalize:

```text
G = (F_flat @ F_flat.T) / (C * H * W)
```

Why normalize?
- deeper/earlier layers have different numbers of channels and spatial positions,
- without normalization, large feature maps can dominate the loss just because they contain more numbers,
- normalization makes style losses from different layers easier to balance.

Style loss for one layer:

```text
L_style_l = mean((G_l(generated) - G_l(style))^2)
```

Style loss across several layers:

```text
L_style = sum over style layers of L_style_l
```

Total objective:

```text
L_total = content_weight * L_content + style_weight * L_style + tv_weight * L_tv
```

The optimizer changes the generated image pixels to reduce this loss.


## 1. Setup

The setup cells import libraries, define configuration, and provide plotting/image helpers. The first run may download VGG19 weights and the default content/style images.


In [ ]:
import random
from io import BytesIO

import numpy as np
import pandas as pd
import requests
from PIL import Image

import torch
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as T
from torchvision.models import vgg19, VGG19_Weights

import matplotlib.pyplot as plt


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device


### Config

The default settings are chosen to finish reasonably quickly on Colab GPU. If runtime is too slow, use `image_size=192` or `steps=80` while debugging, then return to the default settings for the final run if possible.


In [ ]:
CFG = {
    'image_size': 224,
    'steps': 120,
    'lr': 0.03,
    'content_weight': 1.0,
    'style_weight': 1e6,
    'tv_weight': 0.0,
}
CFG


### Helpers

These helpers are provided so the homework can focus on feature maps, losses, and experiments rather than image boilerplate.


In [ ]:
VGG_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
VGG_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def print_shape(name, x):
    if x is None:
        print(f'{name}: not filled yet')
        return
    print(f'{name}: shape={tuple(x.shape)} dtype={x.dtype}')


def load_image_from_url(url, image_size):
    response = requests.get(url, timeout=20)
    response.raise_for_status()
    image = Image.open(BytesIO(response.content)).convert('RGB')
    image = image.resize((image_size, image_size))
    return image


def pil_to_tensor(image):
    return T.ToTensor()(image).unsqueeze(0).to(device)


def vgg_normalize(image_tensor):
    mean = VGG_MEAN.to(image_tensor.device)
    std = VGG_STD.to(image_tensor.device)
    return (image_tensor - mean) / std


def tensor_to_image(image_tensor):
    image = image_tensor.detach().cpu().squeeze(0)
    image = image.clamp(0, 1).permute(1, 2, 0).numpy()
    return image


def show_tensor_image(image_tensor, title=None):
    plt.figure(figsize=(4, 4))
    plt.imshow(tensor_to_image(image_tensor))
    if title is not None:
        plt.title(title)
    plt.axis('off')
    plt.show()


def show_images_side_by_side(items, figsize=(10, 4)):
    plt.figure(figsize=figsize)
    for i, (title, image_tensor) in enumerate(items, start=1):
        plt.subplot(1, len(items), i)
        plt.imshow(tensor_to_image(image_tensor))
        plt.title(title)
        plt.axis('off')
    plt.tight_layout()
    plt.show()


def show_feature_channels(feature_map, title, max_channels=6):
    feature_map = feature_map.detach().cpu()[0]
    n_channels = min(max_channels, feature_map.shape[0])
    plt.figure(figsize=(12, 3))
    for i in range(n_channels):
        plt.subplot(1, n_channels, i + 1)
        plt.imshow(feature_map[i], cmap='gray')
        plt.axis('off')
        plt.title(f'ch {i}')
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


def show_gram_heatmap(gram, title='Gram matrix'):
    plt.figure(figsize=(4, 4))
    plt.imshow(gram.detach().cpu(), cmap='magma')
    plt.title(title)
    plt.xlabel('channel')
    plt.ylabel('channel')
    plt.colorbar()
    plt.show()


### Helper Function Contracts

Use these helpers instead of rewriting image and plotting boilerplate.

| Helper | Use it when | Inputs | Returns |
|---|---|---|---|
| `load_image_from_url(url, image_size)` | You need a default or custom image from a URL | URL string, output size | PIL RGB image |
| `pil_to_tensor(image)` | You need a model-ready image tensor | PIL image | tensor `[1, 3, H, W]` on `device`, values in `[0, 1]` |
| `vgg_normalize(image_tensor)` | You feed an image into VGG | image tensor | ImageNet-normalized tensor |
| `tensor_to_image(image_tensor)` | You need to display a tensor | tensor `[1, 3, H, W]` | NumPy image in `[0, 1]` |
| `show_tensor_image(image_tensor, title=None)` | You need to display one image | image tensor, optional title | displays a figure |
| `show_images_side_by_side(items, figsize=(10, 4))` | You need to compare images | list of `(title, image_tensor)` pairs | displays a figure |
| `show_feature_channels(feature_map, title, max_channels=6)` | You need to inspect VGG activations | feature map `[1, C, H, W]` | displays selected channels |
| `show_gram_heatmap(gram, title='Gram matrix')` | You need to inspect style statistics | Gram matrix `[C, C]` | displays a heatmap |
| `extract_features(image_tensor, model, layers)` | You need VGG activations | image tensor, VGG model, list of layer names | dictionary `{layer_name: feature_map}` |
| `run_style_transfer(...)` | You need to optimize a generated image | init image, weights, steps, lr | generated image tensor and history DataFrame |

Student-implemented functions are `content_loss(...)`, `gram_matrix(...)`, and `style_loss(...)`. The optimization loop is provided.


## 2. Provided Infrastructure: Images and VGG19

We will use the classic neural style transfer pair from the PyTorch tutorial.

This section is mostly provided because URL loading, transforms, and VGG setup are not the main learning goals. If the default URLs fail, you may replace them with your own image URLs or manually uploaded images.

Layer names are strings because we index layers by their position inside `vgg.features`. For example, layer `'0'` is very early, and layer `'21'` is deeper.


In [ ]:
DEFAULT_CONTENT_URL = 'https://pytorch.org/tutorials/_static/img/neural-style/dancing.jpg'
DEFAULT_STYLE_URL = 'https://pytorch.org/tutorials/_static/img/neural-style/picasso.jpg'

content_pil = load_image_from_url(DEFAULT_CONTENT_URL, CFG['image_size'])
style_pil = load_image_from_url(DEFAULT_STYLE_URL, CFG['image_size'])

content_img = pil_to_tensor(content_pil)
style_img = pil_to_tensor(style_pil)

print_shape('content_img', content_img)
print_shape('style_img', style_img)
show_images_side_by_side([
    ('Content image', content_img),
    ('Style image', style_img),
])


In [ ]:
content_layer = '21'
style_layers = ['0', '5', '10', '19', '28']
all_layers = [content_layer] + style_layers

vgg = vgg19(weights=VGG19_Weights.DEFAULT).features.to(device).eval()
for parameter in vgg.parameters():
    parameter.requires_grad = False


def extract_features(image_tensor, model, layers):
    features = {}
    x = vgg_normalize(image_tensor)
    for layer_index, layer in enumerate(model):
        x = layer(x)
        layer_name = str(layer_index)
        if layer_name in layers:
            features[layer_name] = x
    return features


content_features = extract_features(content_img, vgg, all_layers)
style_features = extract_features(style_img, vgg, all_layers)

for layer_name, feature in content_features.items():
    print(layer_name, tuple(feature.shape))


## 3. Exercise 1: Inspect VGG Feature Maps

VGG is fixed, but its intermediate activations give us several ways to describe an image.

Task:
- select one early layer and one deeper layer,
- retrieve their feature maps from `content_features`,
- print their shapes,
- visualize a few channels from each layer,
- decide which layer preserves spatial structure more clearly.

Useful contracts:
- `content_features[layer_name]` returns a feature map with shape `[1, C, H, W]`.
- Early layers usually have larger `H, W` and simpler patterns.
- Deeper layers usually have smaller `H, W` and more semantic patterns.


In [ ]:
# Exercise 1

early_layer = '0'
deep_layer = content_layer

early_features = None
deep_features = None

print_shape('early_features', early_features)
print_shape('deep_features', deep_features)

show_feature_channels(early_features, title=f'Layer {early_layer}')
show_feature_channels(deep_features, title=f'Layer {deep_layer}')


### Checks (Exercise 1)

In [ ]:
assert early_features.ndim == 4
assert deep_features.ndim == 4
assert early_features.shape[2] >= deep_features.shape[2]
assert early_features.shape[3] >= deep_features.shape[3]
print('Exercise 1 passed.')


### Exercise 1 written interpretation

Write 3-5 sentences here:

- How do the early and deeper feature maps differ in shape?
- Which feature maps look more spatially detailed?
- Why might a deeper layer be useful for content loss?


## 4. Exercise 2: Content Loss and Gram Matrix

Content loss compares feature maps directly:

```text
L_content = mean((F_generated - F_content)^2)
```

Style loss starts with a Gram matrix. A Gram matrix summarizes feature-channel co-activation.

Given:

```text
features: [1, C, H, W]
```

For one image:

```text
features[0]: [C, H, W]
flattened : [C, H * W]
gram      : [C, C]
```

Formula:

```text
gram = flattened @ flattened.T / (C * H * W)
```

The entry `gram[i, j]` is large when feature channels `i` and `j` tend to activate together across the image. This is why the Gram matrix can represent texture/style without preserving the exact spatial layout.

Task:
- implement `content_loss`,
- implement `gram_matrix`,
- visualize Gram matrices for an early and deeper style layer.

Useful contracts:
- `content_loss(generated_feat, content_feat)` returns MSE between two feature maps.
- `gram_matrix(features)` receives `[1, C, H, W]` and returns `[C, C]`.
- Use `features[0]` because batch size is 1 in this notebook.
- Use `reshape(C, H * W)` to flatten spatial dimensions.
- Normalize by `C * H * W` so layers with many pixels/channels do not dominate only because of size.


In [ ]:
# Exercise 2

def content_loss(generated_feat, content_feat):
    return None


def gram_matrix(features):
    return None

first_style_feat = style_features[style_layers[0]]
first_style_gram = gram_matrix(first_style_feat)

early_gram = None
deep_gram = None

print_shape('first_style_gram', first_style_gram)
show_gram_heatmap(early_gram, title=f'Gram matrix, layer {style_layers[0]}')
show_gram_heatmap(deep_gram, title=f'Gram matrix, layer {style_layers[-1]}')


### Checks (Exercise 2)

In [ ]:
assert first_style_gram.shape[0] == first_style_gram.shape[1]
assert first_style_gram.shape[0] == first_style_feat.shape[1]
assert torch.allclose(first_style_gram, first_style_gram.T, atol=1e-5)
assert early_gram.ndim == 2
assert deep_gram.ndim == 2
loss_value = content_loss(content_features[content_layer], content_features[content_layer])
assert torch.isclose(loss_value, torch.tensor(0.0, device=loss_value.device))
print('Exercise 2 passed.')


### Exercise 2 written interpretation

Write 3-5 sentences here:

- What shape does your Gram matrix have?
- Why should the Gram matrix be symmetric?
- What visual difference do you notice between early and deeper Gram heatmaps?


## 5. Exercise 3: Style Loss Across Layers

A single layer is not enough for style.

Different VGG layers capture different visual scales:
- early layers: color, edges, fine texture,
- middle layers: repeated patterns,
- deeper layers: larger parts and structure.

So neural style transfer usually compares Gram matrices from several layers.

For each style layer `l`:

```text
G_l_style     = gram(F_l(style))
G_l_generated = gram(F_l(generated))
L_style_l     = mean((G_l_generated - G_l_style)^2)
```

Then:

```text
L_style = sum(L_style_l for l in style_layers)
```

Task:
- precompute target Gram matrices for the style image,
- implement `style_loss`,
- check that the style image has near-zero style loss against itself.

Function contract:
- `style_targets`: dictionary from layer name to target Gram matrix.
- `generated_features`: dictionary from layer name to generated-image feature map.
- `style_layers`: list of layer names used for style.


In [ ]:
# Exercise 3

style_targets = {}

for layer_name in style_layers:
    style_targets[layer_name] = None


def style_loss(generated_features, style_targets, style_layers):
    total = None
    return total

style_self_loss = None
print('style self-loss:', style_self_loss)


### Checks (Exercise 3)

In [ ]:
assert set(style_targets.keys()) == set(style_layers)
for layer_name in style_layers:
    assert style_targets[layer_name].ndim == 2
assert torch.isfinite(style_self_loss)
assert style_self_loss.item() < 1e-8
print('Exercise 3 passed.')


### Exercise 3 written interpretation

Write 2-4 sentences here:

- Why do we use several style layers instead of one?
- Why should the style image have near-zero style loss against its own targets?


## 6. Provided Demo: Baseline Style Transfer

The optimization loop is provided.

Important idea:
- VGG is fixed,
- the generated image is the trainable object,
- every step updates pixels so content/style losses become smaller.

Runtime note: this section is the first expensive part of the notebook. If it is too slow, reduce `CFG['steps']` temporarily while debugging.


In [ ]:
def total_variation_loss(image):
    vertical_diff = torch.abs(image[:, :, 1:, :] - image[:, :, :-1, :]).mean()
    horizontal_diff = torch.abs(image[:, :, :, 1:] - image[:, :, :, :-1]).mean()
    return vertical_diff + horizontal_diff


def run_style_transfer(
    init_image,
    steps,
    content_weight,
    style_weight,
    tv_weight=0.0,
    lr=0.03,
    print_every=40,
):
    generated = init_image.clone().detach().to(device)
    generated.requires_grad_(True)
    optimizer = optim.Adam([generated], lr=lr)
    history = []

    for step in range(steps):
        optimizer.zero_grad()
        generated_features = extract_features(generated, vgg, all_layers)

        c_loss = content_loss(generated_features[content_layer], content_features[content_layer])
        s_loss = style_loss(generated_features, style_targets, style_layers)
        tv_loss = total_variation_loss(generated)
        total_loss = content_weight * c_loss + style_weight * s_loss + tv_weight * tv_loss

        total_loss.backward()
        optimizer.step()

        with torch.no_grad():
            generated.clamp_(0, 1)

        if step == 0 or (step + 1) % print_every == 0 or step + 1 == steps:
            print(
                f'step {step + 1:03d} | '
                f'content {c_loss.item():.4f} | '
                f'style {s_loss.item():.6f} | '
                f'total {total_loss.item():.4f}'
            )

        history.append({
            'step': step + 1,
            'content_loss': float(c_loss.item()),
            'style_loss': float(s_loss.item()),
            'tv_loss': float(tv_loss.item()),
            'total_loss': float(total_loss.item()),
        })

    return generated.detach(), pd.DataFrame(history)


In [ ]:
# Baseline demo

generated_baseline, history_baseline = run_style_transfer(
    init_image=content_img,
    steps=CFG['steps'],
    content_weight=CFG['content_weight'],
    style_weight=CFG['style_weight'],
    tv_weight=CFG['tv_weight'],
    lr=CFG['lr'],
)

show_images_side_by_side([
    ('Content', content_img),
    ('Style', style_img),
    ('Generated', generated_baseline),
], figsize=(12, 4))

history_baseline[['content_loss', 'style_loss', 'total_loss']].plot(figsize=(8, 4), title='Baseline losses')
plt.xlabel('step')
plt.show()


## 7. Exercise 4: Content-Heavy vs Style-Heavy

Now we change only one thing: the style weight.

Task:
- run a content-heavy experiment,
- run a style-heavy experiment,
- compare how much structure and texture each result preserves,
- write a short note in `weight_experiment_note`.

Useful contract:
- larger `style_weight` makes Gram-matrix matching more important,
- smaller `style_weight` leaves more visible content structure.

Keep everything else the same so the comparison is fair.


In [ ]:
# Exercise 4

content_heavy_weight = 2e5
style_heavy_weight = 2e6

content_heavy_img, content_heavy_history = None, None
style_heavy_img, style_heavy_history = None, None

# show_images_side_by_side([
#     ('Content heavy', content_heavy_img),
#     ('Baseline', generated_baseline),
#     ('Style heavy', style_heavy_img),
# ], figsize=(12, 4))

weight_experiment_note = ''
print(weight_experiment_note)


### Checks (Exercise 4)

In [ ]:
assert content_heavy_img.shape == content_img.shape
assert style_heavy_img.shape == content_img.shape
assert len(content_heavy_history) == CFG['steps']
assert len(style_heavy_history) == CFG['steps']
assert isinstance(weight_experiment_note, str) and len(weight_experiment_note) > 0
print('Exercise 4 passed.')


### Exercise 4 written interpretation

Write 3-5 sentences here:

- Which result preserved the content image structure more clearly?
- Which result looked more like the style image texture?
- Did the losses and images tell the same story?


## 8. Exercise 5: Content Init vs Noise Init

The generated image needs an initial value.

Task:
- compare starting from the content image against starting from noise,
- keep all weights the same,
- describe which result is more stable and which looks more stylized,
- write a short note in `init_experiment_note`.

This is a controlled experiment: only the initialization should change.


In [ ]:
# Exercise 5

noise_init = torch.rand_like(content_img)

content_init_img, content_init_history = generated_baseline, history_baseline
noise_init_img, noise_init_history = None, None

show_images_side_by_side([
    ('Content init', content_init_img),
    ('Noise init', noise_init_img),
], figsize=(8, 4))

init_experiment_note = ''
print(init_experiment_note)


### Checks (Exercise 5)

In [ ]:
assert noise_init.shape == content_img.shape
assert content_init_img.shape == content_img.shape
assert noise_init_img.shape == content_img.shape
assert len(noise_init_history) == CFG['steps']
assert isinstance(init_experiment_note, str) and len(init_experiment_note) > 0
print('Exercise 5 passed.')


### Exercise 5 written interpretation

Write 3-5 sentences here:

- Which initialization produced a more stable result?
- Which result looked more stylized?
- Why might starting from the content image make optimization easier?


## 9. Results Summary

A small result table makes the lab easier to discuss.

Record the experiment settings and a short qualitative note. Do not leave the `comment` fields empty in your final submission.


In [ ]:
# Results summary

results_df = pd.DataFrame([
    {'experiment': 'baseline', 'content_weight': CFG['content_weight'], 'style_weight': CFG['style_weight'], 'init': 'content', 'comment': 'balanced baseline'},
    {'experiment': 'content-heavy', 'content_weight': CFG['content_weight'], 'style_weight': content_heavy_weight, 'init': 'content', 'comment': ''},
    {'experiment': 'style-heavy', 'content_weight': CFG['content_weight'], 'style_weight': style_heavy_weight, 'init': 'content', 'comment': ''},
    {'experiment': 'noise-init', 'content_weight': CFG['content_weight'], 'style_weight': CFG['style_weight'], 'init': 'noise', 'comment': ''},
])
results_df


## 10. Optional Demo: Total Variation Loss

Total variation loss encourages neighboring pixels to be similar.

It can make results smoother, but too much smoothing removes texture.


In [ ]:
# Optional demo

# tv_img, tv_history = run_style_transfer(
#     init_image=content_img,
#     steps=CFG['steps'],
#     content_weight=CFG['content_weight'],
#     style_weight=CFG['style_weight'],
#     tv_weight=1e-4,
#     lr=CFG['lr'],
# )
# show_images_side_by_side([
#     ('Baseline', generated_baseline),
#     ('With TV loss', tv_img),
# ], figsize=(8, 4))


## 11. Homework Deliverable

Before submitting, check that your final notebook contains:
- [ ] feature-map inspection for early and deeper VGG layers
- [ ] implemented `content_loss(...)`
- [ ] implemented `gram_matrix(...)` with a symmetric `[C, C]` result
- [ ] implemented multi-layer `style_loss(...)`
- [ ] baseline style-transfer result
- [ ] content-heavy vs style-heavy comparison
- [ ] content-init vs noise-init comparison
- [ ] result summary table with non-empty comments
- [ ] short written interpretations for the main exercises
- [ ] no unresolved `None` placeholders in the required path


## 12. Wrap-Up Questions

Please answer briefly in markdown:

1. In this homework, what exactly is transferred from VGG19?
2. Why do we keep VGG fixed instead of training it?
3. What do early and deeper VGG feature maps seem to capture?
4. Why does the Gram matrix represent style better than raw feature maps?
5. What changed when you increased the style weight?
6. What changed when you initialized from noise instead of the content image?
